In [ ]:
!python --version

# Find Candidates Notebook

I have roughly 14k images split from the video clips captured by the Raspberry Pi and Camera Module. This feels like too many to upload to Roboflow so I'm going to identify frames with bicycles before uploading to Roboflow.

## Basic Steps
 - imports
 - config
 - load the model
 - test on known bicycle containing frames
 - loop over all frames

In [ ]:
from pathlib import Path
from IPython.display import display, Image
from ultralytics import YOLO

In [ ]:
frames_path = Path('frames')

# set confidence quite low, easier to filter false positives than miss false negatives
confidence = 0.05

# using yolo26m model, running on M4 Macbook Air and should perform better than nano
model = YOLO('yolo26m.pt')

In [ ]:
frames = sorted(frames_path.glob('*.jpg'))

# verify we have a frame
display(Image(str(frames[0])))

In [ ]:
# test on a few known bicycle containing frames
test_frames = [
    frames_path / 'clip_20260812_131551_0047.jpg', # NB lane
    frames_path / 'clip_20260812_131551_0289.jpg', # SB lane
    frames_path / 'clip_20260812_131551_0290.jpg', # SB lane
]

for frame in test_frames:
    results = model(str(frame), conf=confidence, classes=[1], iou=0.9, device='mps', verbose=False)
    boxes = results[0].boxes
    print(f'{frame.name}: {len(boxes)} detections')
    for box in boxes:
        conf = float(box.conf[0])
        xyxy = box.xyxy[0].tolist()
        print(f'    conf={conf:.3f}  box={[round(v) for v in xyxy]}')

In [ ]:
import cv2
import supervision as sv

box_annotator = sv.BoxAnnotator()
label_annotator = sv.LabelAnnotator()

def show_detections(image_path, results, width=900):
    """Draw boxes on an image and display it"""
    img = cv2.imread(str(image_path))
    detections = sv.Detections.from_ultralytics(results[0])

    labels = [f"{conf:.2f}" for conf in detections.confidence]

    annotated = box_annotator.annotate(scene=img, detections=detections)
    annotated = label_annotator.annotate(
        scene=annotated, detections=detections, labels=labels
    )

    _, buf = cv2.imencode('.jpg', annotated)
    display(Image(data=buf.tobytes(), width=width))

In [ ]:
for frame in test_frames:
    results = model(str(frame), conf=confidence, classes=[1], verbose=False)
    print(frame.name)
    show_detections(frame, results)

In [ ]:
import random, time

random.seed(18)
sample = random.sample(frames, 1000)

start = time.time()
hits = []
for frame in sample:
    results = model(
        str(frame), conf=confidence, classes=[1], device=device, verbose=False
    )
    if len(results[0].boxes) > 0:
        hits.append((frame, results))
elapsed = time.time() - start
print(f'{device}: {len(hits)}/1000 hits, {elapsed:.1f}s ({elapsed/1000*1000:.0f}ms/frame)')

In [ ]:
for frame, results in hits:
    confs = [round(float(b.conf[0]), 3) for b in results[0].boxes]
    print(f'{frame.name}: {confs}')
    show_detections(frame, results)

## Observations

After reviewing the results from the sample of 1,000 frames, a few things jump out to me.

1. Bikes with crates, bags, etc seem to generate much lower confidence scores, mostly < 0.5. These make up a large number of bikes in Chicago so it's worth considering how to identify these more confidently. Cargo bikes and bikes with large fenders seem to generate even lower confidence scores. These make up less of the bikes on the road but still worth considering.
2. A shadow is boxed with a confidence of 0.39, which is higher than some bikes. False positives like this would lead to an overcount of total bikes and should be prevented if possible.
3. The Divvy guy, who by chance shows up twice in this sample, is boxed initially, then later not boxed. This could present challenges when tracking bikes across a counting line.
4. I expected southbound bikes further from the camera to be harder to identify, but the model seemed to perform better with those relative to the closer northbound bikes that were at a greater angle to the camera.
5. There were several frames where a northbound bike was not identified, even though several others were identified with high confidence scores. The angle relative to the camera combined with features that obscure the "core bike" lead to lower confidence scores, which will be challenging when using this window mounted camera to count both northbound and southbound bikes.
6. The parked scooter handlebars show up consistently, maybe worth considering these as hard negative examples.
   
## Takeaways
1. I should keep the confidence low for initial identification of frames, confidence scores vary widely based on type of bike, fenders, racks, etc.
2. Northbound bikes appear at a greater angle relative to the camera and are sometimes not identified at all. These will require some manual picking of frames to build a sufficient dataset to identify these more irregular bike appearances.
3. This exercise suggests false positives and flicker will be real challenges affecting the outcome of the project, so getting close to the real bike count in test clips needs to be the ultimate metric.

In [ ]:
import json
import shutil

candidates_path = Path('candidates')
if not candidates_path.exists():
    candidates_path.mkdir()
    print(f'Created candidates directory: {candidates_path}')

confidence = 0.05
device = 'mps'

start = time.time()

for frame in frames:
    if (candidates_path / frame.name).exists():
        continue
    results = model(
        str(frame), conf=confidence, classes=[1], device=device, verbose=False
    )
    if len(results[0].boxes) > 0:
        # changed when downgrading to python 3.13
        # frame.copy_into(candidates_path, preserve_metadata=True)
        shutil.copy2(frame, candidates_path)
        boxes = results[0].boxes
        metadata = {
            'source_frame': frame.name,
            'model': 'yolo26m.pt',
            'conf_threshold': confidence,
            'detections': [
                {
                    'confidence': round(float(b.conf[0]), 4),
                    'xyxy': [round(float(v), 1) for v in b.xyxy[0]]
                }
                for b in boxes
            ],
        }
        json_path = candidates_path / f'{frame.stem}.json'
        json_path.write_text(json.dumps(metadata, indent=2))
        

elapsed = time.time() - start
hit_count = len(list(candidates_path.glob('*.jpg')))
print(f'{device}: {hit_count}/{len(frames)} hits, {elapsed:.1f}s ({elapsed/len(frames)*1000:.0f}ms/frame)')

In [ ]:
jpgs = len(list(candidates_path.glob('*.jpg')))
jsons = len(list(candidates_path.glob('*.json')))
print(f'{jpgs} images, {jsons} metadata files')

## Next Steps

The YOLO model pulled out 994 frames out of 14,250, roughly 7% which is in line with the random samples from earlier. This won't include northbound bikes that were not identified at all, which will require manual picking.

From here, I will upload the candidate frames to roboflow to try auto-labeling, and explore labeling using tools locally.